# BRFSS 2015 - classical baselines (LR, SVM, Decision Tree)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laiba-Gul/explainable-diabetes-prediction/blob/main/notebooks/04_brfss_classical_baselines.ipynb)

**Status:** supporting research notebook (original Colab work, cleaned). Outputs were cleared to keep the repository small and free of personal metadata; re-run it or see the reproducible results in [`results/`](../results/).

**What it does:** Trains logistic regression, an RBF SVM (20k-row subsample) and a decision tree on the outputs of notebook 03 (de-duplicated data). Original test ROC-AUC: LR 0.807, SVM 0.799, unpruned tree 0.588.

**Data:** run `diabetes-xai download` (or upload the CSV when the notebook asks). The CSVs are checksum-verified and are not stored in Git.


# Diabetes Prediction — Machine Learning Models

**Models:** Logistic Regression · Support Vector Machine (SVM) · Decision Tree

This notebook picks up **after** preprocessing (cleaning, stratified 70/15/15 split, `StandardScaler` normalization fit on train only, and SMOTE balancing of the training set) is done. It does **not** repeat any of that — it loads the already-prepared CSVs and reconstructs `X_train, X_val, X_test, y_train, y_val, y_test` directly from them.

For each model, this notebook:
1. Trains on the SMOTE-balanced training set only
2. Predicts on the held-out test set
3. Computes Accuracy, Precision, Recall, F1-score, ROC-AUC, MCC, Specificity, Sensitivity, and Log Loss
4. Plots a soft-colored Classification Report, Confusion Matrix, and ROC Curve
5. Adds the model to a shared comparison table

A combined ROC curve (all three models) and a final `ML_Model_Results` comparison table/figures are saved and zipped for download at the end.

**No data leakage:** every model is fit only on `X_train`/`X_train_svm`; `X_val` and `X_test` are only ever passed to `.predict()` / `.predict_proba()`, never to `.fit()`. Scaling and SMOTE were already fit on the training split only, in the preprocessing notebook.


## 1. Setup & Imports

In [ ]:
# If running on a fresh Colab runtime, these are already installed.
# Uncomment only if you hit a "No module named ..." error.
# !pip install -q scikit-learn scipy seaborn matplotlib pandas numpy

import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, log_loss,
    confusion_matrix, classification_report, roc_curve,
)

%matplotlib inline


## 2. Load the Already-Prepared Data (No Preprocessing Repeated)

This assumes the preprocessing notebook already produced three files:

- `train_smote_balanced.csv` — SMOTE-balanced, scaled **training** set
- `val_scaled.csv` — scaled **validation** set (original, untouched class balance)
- `test_scaled.csv` — scaled **test** set (original, untouched class balance)

We train on the SMOTE-balanced training set (models see a balanced sample of both classes) and evaluate on the untouched validation/test sets (so reported metrics reflect real-world class balance, not the synthetic training distribution). Nothing is re-cleaned, re-scaled, or re-split here — that would risk leaking information and duplicates the earlier work.

Upload the three CSVs below (Colab), or edit the paths in the next cell if they're already available in this runtime / a mounted Drive.


In [ ]:
try:
    from google.colab import files
    print("Upload train_smote_balanced.csv, val_scaled.csv, and test_scaled.csv")
    uploaded = files.upload()
except ImportError:
    uploaded = {}
    print("Not running in Colab — set the paths manually in the next cell.")


In [ ]:
TARGET = "Diabetes_binary"

TRAIN_PATH = "train_smote_balanced.csv"
VAL_PATH = "val_scaled.csv"
TEST_PATH = "test_scaled.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

# Reconstruct the six prepared variables. This is a plain feature/target split —
# no scaling, imputing, or resampling happens here (all of that was already
# done, on the training split only, in the preprocessing notebook).
X_train, y_train = train_df.drop(columns=[TARGET]), train_df[TARGET]
X_val, y_val = val_df.drop(columns=[TARGET]), val_df[TARGET]
X_test, y_test = test_df.drop(columns=[TARGET]), test_df[TARGET]

print("X_train:", X_train.shape, " class balance:", y_train.value_counts(normalize=True).round(3).to_dict())
print("X_val:  ", X_val.shape,   " class balance:", y_val.value_counts(normalize=True).round(3).to_dict())
print("X_test: ", X_test.shape,  " class balance:", y_test.value_counts(normalize=True).round(3).to_dict())


## 3. Configuration

In [ ]:
RANDOM_STATE = 42

OUTDIR = "./ML_Model_Results"
FIGDIR = os.path.join(OUTDIR, "figures")
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(FIGDIR, exist_ok=True)

# Soft, muted color per model — used consistently across every plot below.
SOFT_PALETTE = {
    "Logistic Regression": "#8FBCD4",  # soft blue
    "SVM": "#B4A7D6",                  # soft purple
    "Decision Tree": "#A8D5BA",        # soft green
}

CLASS_NAMES = ["No Diabetes", "Diabetes"]


## 4. Helper Functions

Kept modular and reused identically for every model, so results are directly comparable and the research write-up can cite one consistent methodology.


In [ ]:
def compute_metrics(y_true, y_pred, y_proba):
    """Full metric suite for one model's test-set predictions.

    Sensitivity is mathematically identical to Recall for the positive
    class; both are reported explicitly since the requirements list them
    separately (a common convention in clinical/medical ML papers).
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan

    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1_Score": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_proba),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "Specificity": specificity,
        "Sensitivity": sensitivity,
        "Log_Loss": log_loss(y_true, y_proba),
    }


In [ ]:
def plot_confusion_matrix(cm, model_name, color, save_dir=FIGDIR):
    """Soft-colored confusion matrix heatmap for one model."""
    fig, ax = plt.subplots(figsize=(5, 4.5))
    soft_cmap = sns.light_palette(color, as_cmap=True)
    sns.heatmap(cm, annot=True, fmt="d", cmap=soft_cmap, cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                annot_kws={"size": 13, "weight": "bold"},
                linewidths=1, linecolor="white", ax=ax)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_title(f"Confusion Matrix — {model_name}")
    plt.tight_layout()
    fname = f"confusion_matrix_{model_name.lower().replace(' ', '_')}.png"
    plt.savefig(os.path.join(save_dir, fname), bbox_inches="tight")
    plt.show()
    plt.close()


In [ ]:
def plot_classification_report(y_true, y_pred, model_name, color, save_dir=FIGDIR):
    """Prints the standard text report and renders a soft-colored heatmap of it."""
    print(f"\nClassification Report — {model_name}")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))

    report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                                         output_dict=True, zero_division=0)
    report_dict.pop("accuracy", None)
    report_df = pd.DataFrame(report_dict).T[["precision", "recall", "f1-score"]]

    fig, ax = plt.subplots(figsize=(5.5, 3.2))
    soft_cmap = sns.light_palette(color, as_cmap=True)
    sns.heatmap(report_df, annot=True, fmt=".3f", cmap=soft_cmap, cbar=False,
                linewidths=1, linecolor="white", annot_kws={"size": 11}, ax=ax)
    ax.set_title(f"Classification Report — {model_name}")
    plt.tight_layout()
    fname = f"classification_report_{model_name.lower().replace(' ', '_')}.png"
    plt.savefig(os.path.join(save_dir, fname), bbox_inches="tight")
    plt.show()
    plt.close()


In [ ]:
def plot_roc_curve_single(fpr, tpr, auc_score, model_name, color, save_dir=FIGDIR):
    """Soft-colored ROC curve for one model."""
    fig, ax = plt.subplots(figsize=(5.5, 5))
    ax.plot(fpr, tpr, color=color, linewidth=2.2, label=f"{model_name} (AUC = {auc_score:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="#B0B0B0", linewidth=1.2, label="Chance")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve — {model_name}")
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    fname = f"roc_curve_{model_name.lower().replace(' ', '_')}.png"
    plt.savefig(os.path.join(save_dir, fname), bbox_inches="tight")
    plt.show()
    plt.close()


In [ ]:
def train_and_evaluate(model, model_name, X_train, y_train, X_test, y_test, color):
    """Fit -> predict -> metrics -> plots, for one model.

    `model.fit` is only ever called on the training arguments passed in;
    X_test/y_test are only used with `.predict()` / `.predict_proba()`.
    This is the single no-leakage guarantee every model below relies on.
    """
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]  # P(Diabetes = 1)

    metrics = compute_metrics(y_test, y_pred, y_proba)
    metrics["Model"] = model_name

    cm = confusion_matrix(y_test, y_pred)
    plot_confusion_matrix(cm, model_name, color)
    plot_classification_report(y_test, y_pred, model_name, color)

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plot_roc_curve_single(fpr, tpr, metrics["ROC_AUC"], model_name, color)

    roc_data = {"fpr": fpr, "tpr": tpr, "auc": metrics["ROC_AUC"]}
    return model, metrics, roc_data


## 5. Model 1 — Logistic Regression

`class_weight` is intentionally left at its default (`None`): the training set is already class-balanced via SMOTE in the preprocessing notebook, so adding `class_weight="balanced"` here would over-correct.


In [ ]:
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)

log_reg, lr_metrics, lr_roc = train_and_evaluate(
    log_reg, "Logistic Regression",
    X_train, y_train, X_test, y_test,
    color=SOFT_PALETTE["Logistic Regression"],
)


## 6. Model 2 — Support Vector Machine (SVM)

**A practical note on scale.** Kernel SVMs (`SVC`) scale roughly quadratically-to-cubically with the number of training rows, and `probability=True` (needed here for ROC-AUC/Log Loss) adds an internal 5-fold Platt-scaling calibration on top of that. Training directly on the full SMOTE-balanced training set (which can run into the hundreds of thousands of rows) is impractical in a single Colab session.

To stay reproducible in a normal Colab runtime **while still training on correctly-balanced data**, the SVM is fit on a *stratified random subsample* of `X_train`/`y_train` — size controlled by `SVM_MAX_TRAIN_SAMPLES` below. Validation and test sets are untouched; evaluation is still on the full `X_test`/`y_test`. On a beefier runtime, raise `SVM_MAX_TRAIN_SAMPLES` or set it to `None` to use the full set.


In [ ]:
SVM_MAX_TRAIN_SAMPLES = 20000  # set to None to use the full training set (much slower)

if SVM_MAX_TRAIN_SAMPLES is not None and len(X_train) > SVM_MAX_TRAIN_SAMPLES:
    X_train_svm, _, y_train_svm, _ = train_test_split(
        X_train, y_train,
        train_size=SVM_MAX_TRAIN_SAMPLES,
        stratify=y_train,
        random_state=RANDOM_STATE,
    )
    print(f"SVM training on a stratified subsample: {len(X_train_svm):,} of {len(X_train):,} rows.")
else:
    X_train_svm, y_train_svm = X_train, y_train
    print(f"SVM training on the full training set: {len(X_train_svm):,} rows.")


In [ ]:
svm_clf = SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE)

svm_clf, svm_metrics, svm_roc = train_and_evaluate(
    svm_clf, "SVM",
    X_train_svm, y_train_svm, X_test, y_test,
    color=SOFT_PALETTE["SVM"],
)


## 7. Model 3 — Decision Tree

In [ ]:
dt_clf = DecisionTreeClassifier(random_state=RANDOM_STATE)

dt_clf, dt_metrics, dt_roc = train_and_evaluate(
    dt_clf, "Decision Tree",
    X_train, y_train, X_test, y_test,
    color=SOFT_PALETTE["Decision Tree"],
)


## 8. Model Comparison Table

In [ ]:
METRIC_ORDER = ["Accuracy", "Precision", "Recall", "F1_Score", "ROC_AUC",
                "MCC", "Specificity", "Sensitivity", "Log_Loss"]

comparison_df = pd.DataFrame([lr_metrics, svm_metrics, dt_metrics]).set_index("Model")
comparison_df = comparison_df[METRIC_ORDER].round(4)

comparison_path = os.path.join(OUTDIR, "ML_Model_Results.csv")
comparison_df.to_csv(comparison_path)
print(f"Saved comparison table to: {comparison_path}")
comparison_df


## 9. Combined ROC Curve — All Models

In [ ]:
def plot_combined_roc(roc_data_by_model, palette, save_dir=FIGDIR):
    fig, ax = plt.subplots(figsize=(6.5, 6))
    for model_name, data in roc_data_by_model.items():
        ax.plot(data["fpr"], data["tpr"], color=palette[model_name], linewidth=2.2,
                label=f"{model_name} (AUC = {data['auc']:.3f})")
    ax.plot([0, 1], [0, 1], linestyle="--", color="#B0B0B0", linewidth=1.2, label="Chance")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve Comparison — All Models")
    ax.legend(loc="lower right", fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "combined_roc_curve.png"), bbox_inches="tight")
    plt.show()
    plt.close()

roc_data_by_model = {
    "Logistic Regression": lr_roc,
    "SVM": svm_roc,
    "Decision Tree": dt_roc,
}
plot_combined_roc(roc_data_by_model, SOFT_PALETTE)


## 10. Save & Download Results

Everything (comparison CSV + every figure) lives under `ML_Model_Results/`. The cell below zips it and triggers a download on Colab; elsewhere it just prints the local path.


In [ ]:
print(f"All results written to: {OUTDIR}")
for root, _, files_ in os.walk(OUTDIR):
    for fn in files_:
        print(os.path.join(root, fn))

try:
    from google.colab import files
    import shutil
    shutil.make_archive("ML_Model_Results", "zip", OUTDIR)
    files.download("ML_Model_Results.zip")
except ImportError:
    print("Not running in Colab — your files are already saved locally under:", OUTDIR)
